In [ ]:
from pymarc import MARCReader
import pandas as pd
import requests
from dotenv import load_dotenv
import os
from pinecone import Pinecone
import time
import datetime
import json

In [ ]:
# Load environment variables from .env file
load_dotenv()

# Get ISBN API key from environment variables
ISBN_API_KEY = os.getenv('ISBN_API_KEY')
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

In [ ]:
today = datetime.datetime.now()

In [ ]:
def extract_book_info(response_json):
    """
    Extract specific fields from ISBN API response and return as dictionary.
    
    Args:
        response_json (dict): The JSON response from the ISBN API
        
    Returns:
        dict: Dictionary containing extracted book information
    """
    extracted_info = {}
    
    # The response might have the book data nested under 'book' key
    book_data = response_json.get('book', response_json)
    
    # Extract publisher
    extracted_info['publisher'] = book_data.get('publisher', '')
    extracted_info['_id'] = book_data.get('isbn', '')
    
    # Extract synopsis (might be under 'synopsis', 'overview', or 'description')
    extracted_info['synopsis'] = (
        book_data.get('synopsis') or 
        book_data.get('overview') or 
        book_data.get('description') or 
        ''
    )
    
    # Extract title_long
    extracted_info['title'] = book_data.get('title_long', book_data.get('title', ''))
    
    # Extract pages (might be integer or string)
    pages = book_data.get('pages')
    if pages:
        try:
            extracted_info['pages'] = int(pages)
        except (ValueError, TypeError):
            extracted_info['pages'] = pages
    else:
        extracted_info['pages'] = ''
    
    # Extract date_published
    extracted_info['date_published'] = book_data.get('date_published', '')
    
    # Extract dewey_decimal
    #extracted_info['dewey_decimal'] = book_data.get('dewey_decimal', None)
    
    # Extract subjects (might be a list or string)
    subjects = book_data.get('subjects')
    if isinstance(subjects, list):
        extracted_info['subjects'] = subjects
    elif isinstance(subjects, str):
        extracted_info['subjects'] = [subjects]
    else:
        extracted_info['subjects'] = ''
    
    # Extract authors (might be a list or string)
    authors = book_data.get('authors')
    if isinstance(authors, list):
        extracted_info['authors'] = authors
    elif isinstance(authors, str):
        extracted_info['authors'] = [authors]
    else:
        extracted_info['authors'] = ''
    
    return extracted_info


In [ ]:
def get_book_info(isbn):
    # Clean the ISBN (remove any extra characters, spaces, etc.)
    clean_isbn = isbn.replace('-', '').replace(' ', '').strip()
    
    # API request to ISBN database
    url = f"https://api2.isbndb.com/book/{clean_isbn}"
    headers = {
        'User-Agent': 'python-requests/2.28.1',
        'Authorization': ISBN_API_KEY,  # Replace with your actual API key
        'Accept': '*/*'
    }
    
    
    try:
        start_time = time.time()
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            book_info = extract_book_info(response.json())
            end_time = time.time()
            if end_time - start_time > 1:
                return book_info
            else:
                time.sleep(1)
                return book_info
        else:
            print(f"Error Response: {response.text}")
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")


In [ ]:
def check_none_values(record_dict):
    """
    Check each item in a single record dictionary for None values.
    Replaces None values with empty strings and prints what was changed.
    
    Args:
        record_dict (dict): A single record dictionary to check and modify
        
    Returns:
        dict: The modified dictionary with None values replaced by empty strings
    """
    none_count = 0
    
    try:
        for key, value in record_dict.items():
            if value is None or value == '':
                record_dict[key] = '-999'  # Replace None with empty string
            none_count += 1
    except Exception as e:
        print(f"Error checking record: {e}")
        return None

    return record_dict

In [ ]:
index_name = "quickstart-py"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map": {
                "text": "title",
                "text": "authors",
                "text": "subjects",
                "text": "synopsis",
            }
        }
    )

In [ ]:
# Path to your MARC file
marc_file = 'ExportBibJob605495USMarc.001'

In [ ]:
# Import yesterday's records
yesterday = (today - datetime.timedelta(days=1)).strftime('%Y-%m-%d')
yest_filename = f'book_records_{yesterday}.json'

yest_records = []
try:
    with open(yest_filename, 'r') as f:
        yest_records = json.load(f)
    print(f"Loaded {len(yest_records)} records from {yest_filename}")
except FileNotFoundError:
    print(f"No records file found for yesterday ({yest_filename})")
except Exception as e:
    print(f"Error loading yesterday's records: {e}")

In [ ]:
# List to store records
records = []
num_errors = 0

with open(marc_file, 'rb') as file:
    reader = MARCReader(file, to_unicode=True, force_utf8=True)
    for record in reader:
        # Extract fields; use get_fields to handle multiple occurrences
        try:
            isbn = record['020']['a'] if record['020'] else ''

            gather = True
            for yr in yest_records:
                try:
                    if yr['_id'] == isbn:
                        gather = False
                        break
                except Exception as e:
                    continue

            if gather:
                book_info = get_book_info(isbn)

                # Add the record to the list
                records.append(book_info)
            
        except Exception as e:
            print(f"Error processing record {book_info.get('isbn', 'unknown')}: {e}")
            num_errors += 1

        if len(records) >= 4000:
            break

print(f"Number of records processed: {len(records)}")
print(f"Number of errors: {num_errors}")

In [ ]:
# Create filename with date
filename = 'book_records_{}.json'.format(today.strftime('%Y-%m-%d'))

# Save records to JSON file
with open(filename, 'w') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} records to {filename}")

In [ ]:
clean_records = []
for r in records:
    new_r = check_none_values(r)
    if new_r is not None:
        clean_records.append(new_r)

print(len(clean_records))

In [ ]:
# Target the index
dense_index = pc.Index(index_name)

In [ ]:
problem_records = []
count = 0
for i in range(0, len(clean_records), 95):
    try:
        if count < 30:
            dense_index.upsert_records("example-namespace", clean_records[i:i+95])
            count += 1
        else:
            count = 0
            print(f"Sleeping for 60 seconds")
            time.sleep(60)
    except Exception as e:
        print(f"Error upserting records: {e}")
        print(f"Count: {count}")
        problem_records.append(clean_records[i:i+95])

In [ ]:
time.sleep(10)

# View stats for the index
stats = dense_index.describe_index_stats()
print(stats)

In [ ]:
query = "what is a book that has to do with the loch ness monster?"

In [ ]:
# Search the dense index and rerank results
reranked_results = dense_index.search(
    namespace="example-namespace",
    query={
        "top_k": 3,
        "inputs": {
            'text': query
        }
    },
    rerank={
        "model": "bge-reranker-v2-m3",
        "top_n": 3,
        "rank_fields": ["synopsis"]
    }   
)

# Print the reranked results
for hit in reranked_results['result']['hits']:
    print(hit)

In [ ]:
reranked_results.re